In [ ]:
import os
import sys
import shutil
import csv
import time
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# 0. Configuration de l'environnement (Adapté à Jupyter)

In [ ]:
from api_config import configure_google_api_key

# Configurer la clé API Google (charge depuis Colab Secrets ou .env local)
GOOGLE_API_KEY = configure_google_api_key()

# 1. Nettoyage de l'ancien dossier Chroma DB


In [ ]:
persist_dir = "./chroma_minecraft_db"
if os.path.exists(persist_dir):
    print("🧹 Suppression de l'ancienne base vectorielle Chroma...")
    try:
        shutil.rmtree(persist_dir)
        print("✅ Ancienne base supprimée.")
    except Exception as e:
        print(f"⚠️ Erreur lors de la suppression de l'ancienne base : {e}")

# 2. Lecture des fichiers CSV et reconstruction des documents


In [ ]:

print("\n📂 Chargement des données à partir des fichiers CSV...")
docs = []
files_dir = Path.cwd() / "files"

for csv_path in files_dir.rglob("*.csv"):
    relative_path = csv_path.relative_to(files_dir)
    print(f" - Lecture de {relative_path}...")
    
    # Déterminer la source des métadonnées
    if csv_path.name == "wikipedia.csv":
        source = "wikipedia:Minecraft"
    else:
        # Reconstruire l'URL Fandom
        parts = list(relative_path.parts)
        parts[-1] = csv_path.stem
        page_name = "/".join(parts)
        source = f"https://minecraft.fandom.com/fr/wiki/{page_name}"
        
    paragraphs = []
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            next(reader, None) # Sauter l'en-tête "text"
            for row in reader:
                if row and row[0].strip():
                    paragraphs.append(row[0].strip())
                    
        if paragraphs:
            doc_content = "\n\n".join(paragraphs)
            docs.append(Document(
                page_content=doc_content,
                metadata={"source": source}
            ))
    except Exception as e:
        print(f"❌ Erreur lors de la lecture de {csv_path} : {e}")

print(f"\n📚 Total de documents de haut niveau chargés : {len(docs)}")

# 3. Division de texte (Chunking)


In [ ]:

print("\n✂️ Découpage des documents en chunks (RecursiveCharacterTextSplitter)...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)
split_docs = splitter.split_documents(docs)
print(f"🧩 Nombre total de chunks générés : {len(split_docs)}")

# 4. Enrichissement des métadonnées (Meta-Enrichment)


In [ ]:
print("\n🏷️ Enrichissement des chunks avec les en-têtes de sources...")
def enrich_with_source(documents):
    enriched = []
    for d in documents:
        src = d.metadata.get("source", "unknown")
        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"
        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n\n"
            f"{d.page_content}"
        )
        enriched.append(d)
    return enriched

split_docs = enrich_with_source(split_docs)

# 5. Écriture dans Chroma DB (Appel aux Embeddings Gemini)


In [ ]:
print("\n🚀 Indexation dans Chroma DB (Appels à Gemini Embeddings)...")
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    task_type="retrieval_document"
)

# Initialisation de Chroma
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)

batch_size = 30  # Taille de lot (batch size) optimisée
total_chunks = len(split_docs)

for i in range(0, total_chunks, batch_size):
    batch = split_docs[i:i+batch_size]
    print(f" ⏳ Indexation du lot {i//batch_size + 1} / {int(total_chunks/batch_size) + 1} (Chunks {i} à {min(i+batch_size, total_chunks)})...")
    try:
        vectorstore.add_documents(batch)
        # Ajout d'une pause pour éviter les limites de l'API
        time.sleep(3)
    except Exception as e:
        print(f"❌ Erreur critique lors de l'indexation du lot : {e}")
        print("🔄 Pause de 10 secondes et tentative de récupération...")
        time.sleep(10)
        vectorstore.add_documents(batch)

print("\n🎉 BASE VECTORIELLE COMPLÈTEMENT RECONSTRUITE ET ENREGISTRÉE ! 🎉")

# 6. Vérification et requête de la base de données


In [ ]:
print(f"\n📊 Vérification finale : Nombre total de documents dans la base : {vectorstore._collection.count()}")

all_docs = vectorstore.get()
metadatas = all_docs.get('metadatas', [])
sources_saved = set(m.get('source') for m in metadatas if m)

print("\n📍 Sources effectivement indexées :")
for s in sorted(list(sources_saved)):
    print(f" - {s}")